# Spatiotemporal Forecasting with Graph-Enformer (GEnformer)

This notebook demonstrates the usage of the `GEnformer` for spatiotemporal data (data with both temporal sequences and spatial topology).

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from genformer.models import GEnformer

plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
# Dummy spatiotemporal data generation: 4 spatial nodes with distinct periodic patterns
time_steps = 150
num_nodes = 4
x = np.linspace(0, 40, time_steps)
data = np.zeros((time_steps, num_nodes), dtype=np.float32)

# Node 0: Sine wave
data[:, 0] = np.sin(x)
# Node 1: Cosine wave (dependent on Node 0)
data[:, 1] = np.cos(x) + 0.3 * data[:, 0]
# Node 2: Faster sine wave (dependent on Node 1)
data[:, 2] = np.sin(1.5 * x) - 0.2 * data[:, 1]
# Node 3: Modulated wave (dependent on Node 0 and Node 2)
data[:, 3] = np.sin(x) * np.cos(2 * x) + 0.15 * data[:, 0] + 0.15 * data[:, 2]

# Add some noise
data += np.random.normal(0, 0.1, (time_steps, num_nodes)).astype(np.float32)

df = pd.DataFrame(data, columns=['Node_0', 'Node_1', 'Node_2', 'Node_3'])
series = TimeSeries.from_dataframe(df)

train, val = series[:-10], series[-10:]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.flatten()
for i in range(num_nodes):
    train[f'Node_{i}'].plot(ax=axes[i], label=f'Node {i} (Train)')
    axes[i].legend(loc='upper right')
    axes[i].set_ylabel(f'Node {i}')
fig.suptitle('Spatiotemporal Dummy Data (4 Nodes)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Create dummy adjacency matrix (edges) for 4 nodes
edges = torch.tensor([
    [0, 1, 1, 1],
    [1, 0, 1, 0],
    [1, 1, 0, 1],
    [1, 0, 1, 0]
], dtype=torch.float32)

model = GEnformer(
    input_chunk_length=20,
    output_chunk_length=10,
    edges=edges,
    num_nodes=num_nodes,
    num_samples_engression=5,
    n_epochs=10, # Demo
    batch_size=8,
    d_model=64,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=128,
    dropout=0.1
)

model.fit(train)


In [ ]:
from genformer.utils import generate_forecasts

# Generate forecasts using the custom spatial method
# Output shape will be (M, T_out, N, D_gcn) where D_gcn is the latent spatial dimension
# The model expects exactly input_chunk_length as the history window
predictions_tensor = generate_forecasts(
    model=model,
    history=train[-20:], # 20 is the input_chunk_length
    m_samples=30,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# We average over the latent spatial dimension to get the 3 node forecasts
predictions_nodes = predictions_tensor.mean(dim=-1).cpu().numpy() # (M, T_out, N)
